# Create a custom stage in the notebook

This example defines a new pipeline stage directly in the notebook, inserts it between existing stages, and stores a small ATAR hit summary in the pipeline storage.


## Bootstrap project imports


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents):
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate reco_algorithm_tests project root.")

src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"Project root: {PROJECT_ROOT}")


Project root: /workdir/playground/reco_algorithm_tests


## Import pipeline tools


In [2]:
from algorithms import DefaultTrackletFormer
from data_io import RecoDataFile, load_pioneer_libraries
from pipeline.pipeline import Pipeline
from pipeline.stages import EventInitStage, EventInputStage, InputContext, Stage, TrackletStage


## Load PIONEER libraries and the smoke-test data


In [3]:
load_pioneer_libraries()

DATA_PATH = PROJECT_ROOT / ".data" / "current" / "all_rec.root"
ENTRY_INDEX = 0

reco_data = RecoDataFile(str(DATA_PATH))
print(f"Loaded data file: {DATA_PATH}")
print(f"Entries available: {reco_data.entries}")


Loaded data file: /workdir/playground/reco_algorithm_tests/.data/current/all_rec.root
Entries available: 727


## Define a new stage inline


In [4]:
class AtarHitSummaryStage(Stage):
    def __init__(self):
        super().__init__(
            name="Summarize ATAR Hits",
            stage_key="atar_hit_summary",
            prerequisites=["event_init"],
        )

    def build_handler(self):
        def handler(storage):
            hits = storage["raw_hits"]
            summary = {
                "total_hits": len(hits),
                "front_hits": sum(hit.detector_side == "front" for hit in hits),
                "back_hits": sum(hit.detector_side == "back" for hit in hits),
            }
            storage["atar_hit_summary"] = summary
            storage["event"].extra_info["atar_hit_summary"] = summary
        return handler


## Insert the new stage into a pipeline


In [5]:
pipeline = Pipeline()
pipeline.register_stage(EventInputStage())
pipeline.register_stage(EventInitStage())
pipeline.register_stage(AtarHitSummaryStage())
pipeline.register_stage(TrackletStage(DefaultTrackletFormer()))

pipeline.run(InputContext(reco_data, ENTRY_INDEX))
event = pipeline.get_event()
summary = pipeline.get_storage()["atar_hit_summary"]
summary


{'total_hits': 33, 'front_hits': 16, 'back_hits': 17}

## Inspect the stored stage output


In [6]:
print("Stored summary:", summary)
print("Event extra_info summary:", event.extra_info.get("atar_hit_summary"))
pipeline.status()


Stored summary: {'total_hits': 33, 'front_hits': 16, 'back_hits': 17}
Event extra_info summary: {'total_hits': 33, 'front_hits': 16, 'back_hits': 17}
Pipeline Status (generated in 0.0000s):
  Total stages: 4
  Completed: 4
  Current stage: tracklets
  Next stage: None
  All completed: True
  Storage keys: ['event_data', 'truth_data', 'geo', 'extra_info', 'entry_index', 'event_entry', 'raw_hits', 'reference_truth_entry', 'patterns', 'tracklets', 'vertices', 'event', 'atar_hit_summary']

Stage Details:
Load Event Input Data: done  runnable
Initialize Event: done  runnable
Summarize ATAR Hits: done  runnable
Form Tracklets: done current runnable
